# Industry Risk Analysis: Results Walkthrough

This notebook is the portfolio-facing summary of the current generated outputs. It complements the formal report by reading the live CSV outputs and restating the current headline results.

Current snapshot from the live executive summary:

- Highest industry base risk: Agriculture, Forestry And Fishing at 3.50
- Lowest industry base risk: Transport, Postal And Warehousing at 2.14
- Strongest employment growth: Professional, Scientific And Technical Services at +5.5% YoY
- Weakest employment growth: Wholesale Trade at -8.7% YoY
- Highest borrower archetype score: Agriculture, Forestry And Fishing Archetype at 3.09
- Highest concentration utilisation: Retail Trade at 113.3% of limit
- Concentration breaches: Retail Trade and Wholesale Trade
- Highest AR, AP, inventory, scorecard-overlay, PD-overlay, and LGD-overlay sector: Manufacturing
- Most watchlist triggers: Agriculture, Forestry And Fishing with 5
- Largest average stress uplift: Demand shock at 0.23 score points

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
pd.options.display.max_columns = 100

def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'src').exists() and (candidate / 'output' / 'tables').exists():
            return candidate
    raise RuntimeError('Could not locate repository root.')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

TABLES = REPO_ROOT / 'output' / 'tables'
base = pd.read_csv(TABLES / 'industry_base_risk_scorecard.csv')
borrower = pd.read_csv(TABLES / 'borrower_industry_risk_scorecard.csv')
concentration = pd.read_csv(TABLES / 'concentration_limits.csv')
stress = pd.read_csv(TABLES / 'industry_stress_test_matrix.csv')
watchlist = pd.read_csv(TABLES / 'watchlist_triggers.csv')
industry_wc = pd.read_csv(TABLES / 'industry_working_capital_risk_metrics.csv')
borrower_wc = pd.read_csv(TABLES / 'borrower_working_capital_risk_metrics.csv')
chart_catalog = pd.read_csv(TABLES / 'chart_table.csv')

In [ ]:
watchlist_counts = watchlist.groupby('industry').size().sort_values(ascending=False)
avg_stress = stress.groupby('scenario_name', as_index=False)['stress_delta'].mean().sort_values('stress_delta', ascending=False)

headline = pd.DataFrame([
    ['Highest base risk', base.sort_values('industry_base_risk_score', ascending=False).iloc[0]['industry'], base['industry_base_risk_score'].max()],
    ['Lowest base risk', base.sort_values('industry_base_risk_score').iloc[0]['industry'], base['industry_base_risk_score'].min()],
    ['Strongest employment growth', base.sort_values('employment_yoy_growth_pct', ascending=False).iloc[0]['industry'], base['employment_yoy_growth_pct'].max()],
    ['Weakest employment growth', base.sort_values('employment_yoy_growth_pct').iloc[0]['industry'], base['employment_yoy_growth_pct'].min()],
    ['Highest borrower score', borrower.sort_values('final_industry_risk_score', ascending=False).iloc[0]['borrower_name'], borrower['final_industry_risk_score'].max()],
    ['Highest concentration utilisation', concentration.sort_values('utilisation_pct', ascending=False).iloc[0]['industry'], concentration['utilisation_pct'].max()],
    ['Highest AR benchmark', industry_wc.sort_values('ar_days_benchmark', ascending=False).iloc[0]['industry'], industry_wc['ar_days_benchmark'].max()],
    ['Highest AP benchmark', industry_wc.sort_values('ap_days_benchmark', ascending=False).iloc[0]['industry'], industry_wc['ap_days_benchmark'].max()],
    ['Highest inventory benchmark', industry_wc.sort_values('inventory_days_benchmark', ascending=False).iloc[0]['industry'], industry_wc['inventory_days_benchmark'].max()],
    ['Highest WC PD overlay', industry_wc.sort_values('working_capital_pd_overlay_score', ascending=False).iloc[0]['industry'], industry_wc['working_capital_pd_overlay_score'].max()],
    ['Highest borrower WC PD metric', borrower_wc.sort_values('working_capital_pd_metric_score', ascending=False).iloc[0]['borrower_name'], borrower_wc['working_capital_pd_metric_score'].max()],
    ['Most watchlist triggers', watchlist_counts.index[0], watchlist_counts.iloc[0]],
    ['Largest average stress uplift', avg_stress.iloc[0]['scenario_name'], avg_stress.iloc[0]['stress_delta']],
], columns=['metric', 'leader', 'value'])

display(headline)
display(chart_catalog)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

sector_plot = base.sort_values('industry_base_risk_score')
axes[0, 0].barh(sector_plot['industry'], sector_plot['industry_base_risk_score'], color='#12324a')
axes[0, 0].set_title('Industry Base Risk Score by Sector')

borrower_plot = borrower.sort_values('final_industry_risk_score')
axes[0, 1].barh(borrower_plot['borrower_name'], borrower_plot['final_industry_risk_score'], color='#7b341e')
axes[0, 1].set_title('Borrower Industry Risk Scorecard')

conc_plot = concentration.sort_values('utilisation_pct')
conc_colors = ['#c53030' if x else '#2b6cb0' for x in conc_plot['breach']]
axes[1, 0].barh(conc_plot['industry'], conc_plot['utilisation_pct'], color=conc_colors)
axes[1, 0].axvline(100, color='black', linestyle='--', linewidth=1)
axes[1, 0].set_title('Sector Concentration Utilisation')

wc_plot = industry_wc.sort_values('working_capital_pd_overlay_score')
axes[1, 1].barh(wc_plot['industry'], wc_plot['working_capital_pd_overlay_score'], color='#805ad5')
axes[1, 1].set_title('Working-Capital PD Overlay Score')

plt.tight_layout()
plt.show()

In [ ]:
display(base[['industry', 'classification_risk_score', 'macro_risk_score', 'industry_base_risk_score', 'industry_base_risk_level', 'employment_yoy_growth_pct']].sort_values('industry_base_risk_score', ascending=False))
display(concentration.sort_values('utilisation_pct', ascending=False))
display(industry_wc[['industry', 'ar_days_benchmark', 'ap_days_benchmark', 'inventory_days_benchmark', 'inventory_stock_build_risk', 'working_capital_scorecard_overlay_score', 'working_capital_pd_overlay_score', 'working_capital_lgd_overlay_score']].sort_values('working_capital_pd_overlay_score', ascending=False))
display(watchlist.sort_values(['industry', 'trigger']))

## Refresh

To refresh the notebook after the data or logic changes:

1. Run `python scripts/run_pipeline.py`
2. Re-run this notebook
3. For the derivation logic behind these tables, open `02_methodology_and_output_map.ipynb`

Formal report files:
- `output/executive_summary.md`
- `output/chart_explanations.md`
- `industry_risk_formal_report.pdf`